In [11]:
# Data manipulation
import stackstac
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import mapping, box

# I/O operations
import planetary_computer as pc
from pystac_client import Client as pystac_client

# Dask
import dask
import dask.delayed as delayed
#import dask.dataframe as dd
import dask_geopandas as dask_gpd
from dask.distributed import Client as dask_client
from dask.diagnostics import ProgressBar
from tqdm.notebook import tqdm

# Data Visualization
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

#import warnings
#warnings.filterwarnings("ignore")

In [18]:
train_df = gpd.read_file("data/train.csv")


In [26]:
# Read file
train_df = gpd.read_file("data/train.csv")
train_df["geometry"] = gpd.GeoSeries.from_wkt(train_df["geometry"])
train_df["geojson"] = train_df["geometry"].apply(lambda x: mapping(x))

In [32]:
def grid_partition(region: gpd.GeoDataFrame, cell_size):
    min_x, min_y, max_x, max_y = [round(coord) for coord in region.total_bounds.tolist()]
    x_cells = int((max_x - min_x) / cell_size)
    y_cells = int((max_y - min_y) / cell_size)
    polygons = []
    tile_ids = []
    for i in range(x_cells):
        for j in range(y_cells):

            x1 = min_x + i * cell_size
            y1 = min_y + j * cell_size

            poly = box(x1, y1, x1 + cell_size, y1 + cell_size)
            tile_id = f"{x1}_{y1 + cell_size}"
            
            polygons.append(poly)
            tile_ids.append(tile_id)

    return gpd.GeoDataFrame({"tile_id": tile_ids, "geometry": polygons}, crs="EPSG:4326")

In [52]:
telangana_gdf = gpd.read_file("data/shp/telangana.shp")
tiles = grid_partition(telangana_gdf, 0.5)
tiles

,tile_id,geometry
0,77.0_16.5,"POLYGON ((77.5 16, 77.5 16.5, 77 16.5, 77 16, ..."
1,77.0_17.0,"POLYGON ((77.5 16.5, 77.5 17, 77 17, 77 16.5, ..."
2,77.0_17.5,"POLYGON ((77.5 17, 77.5 17.5, 77 17.5, 77 17, ..."
3,77.0_18.0,"POLYGON ((77.5 17.5, 77.5 18, 77 18, 77 17.5, ..."
4,77.0_18.5,"POLYGON ((77.5 18, 77.5 18.5, 77 18.5, 77 18, ..."
...,...,...
75,81.5_18.0,"POLYGON ((82 17.5, 82 18, 81.5 18, 81.5 17.5, ..."
76,81.5_18.5,"POLYGON ((82 18, 82 18.5, 81.5 18.5, 81.5 18, ..."
77,81.5_19.0,"POLYGON ((82 18.5, 82 19, 81.5 19, 81.5 18.5, ..."
78,81.5_19.5,"POLYGON ((82 19, 82 19.5, 81.5 19.5, 81.5 19, ..."


In [53]:
# Join plots to 0.5° grid cells
train_gdf = gpd.GeoDataFrame(train_df, crs="EPSG:4326")
joined_gdf = train_gdf.sjoin(tiles)

In [54]:
# Load into a Dask dataframe and spatially partition by grid cell
unique_values = sorted(joined_gdf['tile_id'].unique())
divisions = unique_values + [unique_values[-1]]
joined_ddf = dask_gpd.from_geopandas(joined_gdf)
train_ddf = joined_ddf.set_index("tile_id", divisions=divisions)

In [55]:
train_ddf.map_partitions(lambda x: len(x)).compute()

0      18
1      27
2     724
3       1
4      53
5      90
6     449
7     361
8     156
9     220
10    724
11     64
12    662
13    110
14     57
15     36
16     23
17    125
18     10
19    774
20    600
21    552
22    597
23    117
24    136
25    418
26    201
27    607
28      9
dtype: int64